# Task 2 — CLM-8B trên Kaggle (Hướng 2, chạy đầy đủ)

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Dán file id Google Drive của bộ dữ liệu vào biến `DRIVE_ID` ở cell cấu hình (chỉ trên Kaggle, không commit vào repo vì repo công khai) — hoặc dùng secret `PUBLIC_TEST_DRIVE_ID`. Tuỳ chọn `GITHUB_TOKEN` nếu repo private, `HF_TOKEN` nếu cần.
3. Tuỳ chọn: thêm Kaggle dataset chứa `mutants.json` (output của `scripts/augment_task2_api.py`) và sửa `AUGMENT_API_PATH` bên dưới.
4. Run All. Dữ liệu sinh viên và repo nằm trong `/kaggle/tmp` (không được lưu cùng notebook version). Notebook ra cả `task1_predictions.json` (Task 1 theo rubric, probe Ridge trên embedding CLM, không dùng API). Các file cần tải từ Output: `task2_predictions.json`, `task1_predictions.json`, `task2_benchmark.md`.

**Chế độ (`MODE` ở cell cấu hình):**
- `'zeroshot'` (mặc định): CLM-8B không huấn luyện, trả lời câu hỏi có/không (Task 2) và câu hỏi chấm điểm (Task 1) bằng prompt; ngưỡng lấy từ tần suất nhãn của 32 bài gold. Chạy `run_task2_zeroshot.py` + `run_task1_zeroshot.py`. Xem `docs/task2_zeroshot.md`.
- `'probe'`: huấn luyện head trên 32 bài gold + mẫu chèn lỗi (`run_task2_clm.py` + `run_task1_clm.py`), có thêm `task2_benchmark.md`.

Không dùng nhãn sinh trên tập test (quy định của giảng viên); train = 32 bài gold + mẫu chèn lỗi.

Tự compile (giảng viên cho phép 2026-10-11): pipeline chỉ chạy `g++ -std=c++11` (`-fsyntax-only` cho đề nhiều câu) để lấy compile log làm đặc trưng, **không chạy** code sinh viên. Kaggle dùng g++ native; phiên bản có thể khác GCC 13.3 của bản local nên log khác chữ đôi chút (cache tách theo `compiler_id`). Cell cài đặt in `g++ --version` để kiểm tra.


In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-fewshot-clm'
AUGMENT_API_PATH = ''  # optional mutants.json (Kaggle dataset) from scripts/augment_task2_api.py
DRIVE_ID = ''  # optional: Google Drive id (leave blank if dataset is uploaded to /kaggle/input)
WORK = '/kaggle/tmp'  # student data + repo live here, not in /kaggle/working
MODE = 'probe'  # 'probe': train Few-Shot Prototypical/Linear Probe on 32 gold | 'zeroshot': no training

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if os.path.exists(REPO_DIR):
    print('Updating existing repo to latest commit on', BRANCH)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print('Cloning repo on branch', BRANCH)
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print('Current commit:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())


In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' accelerate safetensors scikit-learn pyyaml bitsandbytes
!g++ --version

In [ ]:
import os, shutil
from pathlib import Path

print('==> Scanning /kaggle/input for datasets:')
!ls -la /kaggle/input

# 1. Unpack from /kaggle/input
!python scripts/unpack_data.py /kaggle/input --dest data/raw

# 2. Fallback search: if directories were already extracted by Kaggle
raw = Path('data/raw')
raw.mkdir(parents=True, exist_ok=True)
for name in ['sample_dataset', 'public_test_release']:
    target = raw / name
    if not target.exists():
        for found in Path('/kaggle/input').rglob(name):
            if found.is_dir() and found.resolve() != target.resolve():
                print(f'Copying {found} -> {target}')
                shutil.copytree(found, target)
                break

# 3. Verify datasets exist
assert (raw / 'sample_dataset').exists(), 'LỖI: Chưa có sample_dataset trong data/raw! Hãy kiểm tra dataset đã upload vào Kaggle Input chưa.'
assert (raw / 'public_test_release').exists(), 'LỖI: Chưa có public_test_release trong data/raw! Hãy kiểm tra dataset đã upload vào Kaggle Input chưa.'
print('==> Dữ liệu đã sẵn sàng trong data/raw:')
!ls -la data/raw


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, subprocess
if not os.path.exists('data/raw/sample_dataset'):
    print('==> Auto-unpacking datasets into data/raw...')
    subprocess.run(['python', 'scripts/unpack_data.py', '/kaggle/input', '--dest', 'data/raw'], check=False)
CFG = ['--config', 'configs/task2/clm_kaggle.yaml']
if MODE == 'zeroshot':
    subprocess.run(['python', 'scripts/run_task2_zeroshot.py', *CFG], check=True)
    subprocess.run(['python', 'scripts/run_task1_zeroshot.py', *CFG], check=True)
else:
    args = CFG + (['--augment-api', AUGMENT_API_PATH] if AUGMENT_API_PATH else [])
    subprocess.run(['python', 'scripts/run_task2_clm.py', *args], check=True)
    subprocess.run(['python', 'scripts/run_task1_clm.py', *CFG], check=True)

In [ ]:
import json, shutil
if MODE == 'zeroshot':
    t2_src = 'outputs/task2/task2_zeroshot_clm_kaggle/predictions.json'
    t1_src = 'outputs/task1/task1_zeroshot_clm_kaggle/predictions.json'
else:
    subprocess.run(['python', 'scripts/task2_benchmark.py'], check=True)
    shutil.copy('outputs/task2/benchmark.md', '/kaggle/working/task2_benchmark.md')
    t2_src = 'outputs/task2/task2_clm_kaggle/predictions.json'
    t1_src = 'outputs/task1/task1_clm_kaggle/predictions.json'
shutil.copy(t2_src, '/kaggle/working/task2_predictions.json')
shutil.copy(t1_src, '/kaggle/working/task1_predictions.json')
from src.task2.evaluator import validate_predictions
from src.task1.evaluator import validate_predictions as validate_task1
sub = json.load(open('/kaggle/working/task2_predictions.json'))
print('task2', len(sub), validate_predictions(sub))
sub1 = json.load(open('/kaggle/working/task1_predictions.json'))
print('task1', len(sub1), validate_task1(sub1))